In [ ]:
import numpy as np
print(np.__version__)

import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

import math

In [ ]:
class TicTacToe:
    def __init__(self):
        self.row_count = 3
        self.column_count = 3
        self.action_size = self.row_count * self.column_count

    def __repr__(self):
        return "TicTacToe"

    def get_initial_state(self):
        return np.zeros((self.row_count, self.column_count))

    def get_next_state(self, state, action, player):
        row = action // self.column_count
        column = action % self.column_count
        state[row, column] = player
        return state

    def get_valid_moves(self, state):
        return (state.reshape(-1) == 0).astype(np.uint8)

    def check_win(self, state, action):
        if action == None:
            return False

        row = action // self.column_count
        column = action % self.column_count
        player = state[row, column]

        return (
            np.sum(state[row, :]) == player * self.column_count
            or np.sum(state[:, column]) == player * self.row_count
            or np.sum(np.diag(state)) == player * self.row_count
            or np.sum(np.diag(np.flip(state, axis=0))) == player * self.row_count
        )

    def get_value_and_terminated(self, state, action):
        if self.check_win(state, action):
            return 1, True
        if np.sum(self.get_valid_moves(state)) == 0:
            return 0, True
        return 0, False

    def get_opponent(self, player):
        return -player

    def get_opponent_value(self, value):
        return -value

    def change_perspective(self, state, player):
        return state * player

    def get_encoded_state(self, state):
        encoded_state = np.stack(
            (state == -1, state == 0, state == 1)
        ).astype(np.float32)

        if len(state.shape) == 3:
            encoded_state = np.swapaxes(encoded_state, 0, 1)

        return encoded_state

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

In [ ]:
class RepresentationNetwork(nn.Module):
    """h(observation) -> hidden_state
    Encodes the raw game observation into a latent representation.
    This replaces AlphaZero's direct use of the board state."""

    def __init__(self, game, num_resBlocks, num_hidden):
        super().__init__()
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )
        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resBlocks)]
        )

    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
        return x


class DynamicsNetwork(nn.Module):
    """g(hidden_state, action) -> (next_hidden_state, reward)
    Predicts the next latent state and reward for a given action.
    This replaces AlphaZero's use of game.get_next_state()."""

    def __init__(self, game, num_resBlocks, num_hidden):
        super().__init__()
        self.action_size = game.action_size
        self.row_count = game.row_count
        self.column_count = game.column_count

        self.startBlock = nn.Sequential(
            nn.Conv2d(num_hidden + game.action_size, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )
        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resBlocks)]
        )
        self.rewardHead = nn.Sequential(
            nn.Conv2d(num_hidden, 1, kernel_size=1),
            nn.Flatten(),
            nn.Linear(game.row_count * game.column_count, 1),
            nn.Tanh()
        )

    def forward(self, hidden_state, action):
        batch_size = hidden_state.size(0)

        # Encode action as one-hot spatial planes
        action_one_hot = torch.zeros(batch_size, self.action_size, device=hidden_state.device)
        action_one_hot.scatter_(1, action.long().unsqueeze(1), 1.0)
        action_planes = action_one_hot.unsqueeze(-1).unsqueeze(-1).expand(
            -1, -1, self.row_count, self.column_count
        )

        x = torch.cat([hidden_state, action_planes], dim=1)
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)

        next_hidden_state = self._normalize(x)
        reward = self.rewardHead(x)
        return next_hidden_state, reward

    def _normalize(self, x):
        x_flat = x.view(x.size(0), -1)
        x_min = x_flat.min(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
        x_max = x_flat.max(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
        scale = x_max - x_min
        scale = torch.where(scale < 1e-5, torch.ones_like(scale), scale)
        return (x - x_min) / scale


class PredictionNetwork(nn.Module):
    """f(hidden_state) -> (policy, value)
    Predicts the policy and value from a latent state.
    Same role as AlphaZero's policy+value heads."""

    def __init__(self, game, num_hidden):
        super().__init__()
        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.row_count * game.column_count, game.action_size)
        )
        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_count * game.column_count, 1),
            nn.Tanh()
        )

    def forward(self, hidden_state):
        return self.policyHead(hidden_state), self.valueHead(hidden_state)


class MuZeroNetwork(nn.Module):
    """Wraps all three MuZero networks into a single module.
    Provides initial_inference (for real observations) and
    recurrent_inference (for planning inside MCTS)."""

    def __init__(self, game, num_resBlocks, num_hidden):
        super().__init__()
        self.representation = RepresentationNetwork(game, num_resBlocks, num_hidden)
        self.dynamics = DynamicsNetwork(game, num_resBlocks, num_hidden)
        self.prediction = PredictionNetwork(game, num_hidden)

    def initial_inference(self, observation):
        hidden_state = self.representation(observation)
        policy, value = self.prediction(hidden_state)
        return hidden_state, policy, value

    def recurrent_inference(self, hidden_state, action):
        next_hidden_state, reward = self.dynamics(hidden_state, action)
        policy, value = self.prediction(next_hidden_state)
        return next_hidden_state, reward, policy, value

In [ ]:
class Node:
    def __init__(self, prior=0):
        self.prior = prior
        self.hidden_state = None
        self.reward = 0
        self.visit_count = 0
        self.value_sum = 0
        self.children = {}

    def expanded(self):
        return len(self.children) > 0

    def value(self):
        if self.visit_count == 0:
            return 0
        return self.value_sum / self.visit_count


class MCTS:
    def __init__(self, game, model, args):
        self.game = game
        self.model = model
        self.args = args

    def get_ucb_score(self, parent, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value() + 1) / 2)
        return q_value + self.args['C'] * (math.sqrt(parent.visit_count) / (child.visit_count + 1)) * child.prior

    def select_child(self, node):
        best_score = -np.inf
        best_action = -1
        best_child = None
        for action, child in node.children.items():
            score = self.get_ucb_score(node, child)
            if score > best_score:
                best_score = score
                best_action = action
                best_child = child
        return best_action, best_child

    @torch.no_grad()
    def search(self, state):
        # Initial inference: encode the real observation
        encoded = self.game.get_encoded_state(state)
        tensor = torch.tensor(encoded, dtype=torch.float32).unsqueeze(0)
        hidden_state, policy_logits, _ = self.model.initial_inference(tensor)

        root = Node()
        root.hidden_state = hidden_state
        root.visit_count = 1

        policy = torch.softmax(policy_logits, dim=1).squeeze(0).cpu().numpy()

        # Add Dirichlet noise at the root
        policy = (1 - self.args['dirichlet_epsilon']) * policy + self.args['dirichlet_epsilon'] \
            * np.random.dirichlet([self.args['dirichlet_alpha']] * self.game.action_size)

        # Mask invalid moves (we only know valid moves at the root from the real state)
        valid_moves = self.game.get_valid_moves(state)
        policy *= valid_moves
        policy /= np.sum(policy)

        # Expand root node
        for action in range(self.game.action_size):
            if policy[action] > 0:
                root.children[action] = Node(prior=policy[action])

        # Run simulations
        for _ in range(self.args['num_searches']):
            node = root
            search_path = [node]

            # SELECT: traverse tree until we find an unexpanded node
            while node.expanded():
                action, node = self.select_child(node)
                search_path.append(node)

            parent = search_path[-2]

            # EXPAND: use dynamics network to compute hidden state
            action_tensor = torch.tensor([action], dtype=torch.long)
            hidden_state, reward, policy_logits, value = self.model.recurrent_inference(
                parent.hidden_state, action_tensor
            )

            node.hidden_state = hidden_state
            node.reward = reward.item()

            policy = torch.softmax(policy_logits, dim=1).squeeze(0).cpu().numpy()
            value = value.item()

            # Create children for this node
            for a in range(self.game.action_size):
                if policy[a] > 0:
                    node.children[a] = Node(prior=policy[a])

            # BACKPROPAGATE
            for bnode in reversed(search_path):
                bnode.visit_count += 1
                bnode.value_sum += value
                value = -value  # flip value for opponent

        # Return action probabilities based on visit counts
        action_probs = np.zeros(self.game.action_size)
        for action, child in root.children.items():
            action_probs[action] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs

In [ ]:
import matplotlib.pyplot as plt

tictactoe = TicTacToe()

model = MuZeroNetwork(tictactoe, 4, 64)
model.eval()

args = {
    'C': 2,
    'num_searches': 100,
    'dirichlet_epsilon': 0.25,
    'dirichlet_alpha': 0.3,
}

mcts = MCTS(tictactoe, model, args)

# Run MCTS on the initial state
state = tictactoe.get_initial_state()
action_probs = mcts.search(state)

print("MCTS action probabilities (untrained model):")
print(np.round(action_probs, 3))
print(f"Best action: {np.argmax(action_probs)}")

plt.bar(range(tictactoe.action_size), action_probs)
plt.title("MuZero MCTS Action Probabilities (untrained)")
plt.xlabel("Action")
plt.ylabel("Probability")
plt.show()

In [ ]:
tictactoe = TicTacToe()
player = 1

args = {
    'C': 2,
    'num_searches': 100,
    'dirichlet_epsilon': 0.,
    'dirichlet_alpha': 0.3,
}

model = MuZeroNetwork(tictactoe, 4, 64)
model.eval()

mcts = MCTS(tictactoe, model, args)

state = tictactoe.get_initial_state()

while True:
    print(state)

    if player == 1:
        valid_moves = tictactoe.get_valid_moves(state)
        print("valid_moves", [i for i in range(tictactoe.action_size) if valid_moves[i] == 1])
        action = int(input(f"{player}:"))

        if valid_moves[action] == 0:
            print("action not valid")
            continue

    else:
        neutral_state = tictactoe.change_perspective(state, player)
        mcts_probs = mcts.search(neutral_state)
        action = np.argmax(mcts_probs)

    state = tictactoe.get_next_state(state, action, player)

    value, is_terminal = tictactoe.get_value_and_terminated(state, action)

    if is_terminal:
        print(state)
        if value == 1:
            print(player, "won")
        else:
            print("draw")
        break

    player = tictactoe.get_opponent(player)